# OAuth server

> Let apps such as Claude sign in to a FastHTML app and act as its users

In [ ]:
#| default_exp oauth_server

`OAuthServer` lets apps such as Claude act as the signed-in users of a FastHTML app. An app that calls a protected route without a token, usually an MCP endpoint at `/mcp`, gets a 401 that points to this server. The app registers itself and sends the person to an Allow page. When the person allows it, the app trades a one-time code for an access token. This is OAuth 2.1 as the [MCP authorization spec](https://modelcontextprotocol.io/specification/2025-11-25/basic/authorization) asks for, with the FastHTML app as its own authorization server. Every sign-in uses PKCE with S256.

The server stores nothing by default. It signs each client id, code and token with the app's `secret_key`, as FastHTML signs session cookies. An access token then lasts 30 days. Only `token_user` can end it early, for example after the user signs out of all devices.

Passing a `store` turns on grants, one stored row for each app a user connects. With grants, an access token lasts an hour and a refresh token renews it. A refresh token works once. Using one again disconnects its app. A code works once too. Users can list their connected apps and disconnect each one.

In [ ]:
import base64, hashlib, re, secrets, time
from urllib.parse import urlencode, urlsplit
from itsdangerous import URLSafeTimedSerializer, BadData
from fasthtml.common import *
from fasthtml.oauth import _arun, redir_url

In [ ]:
from urllib.parse import parse_qsl
from bs4 import BeautifulSoup
from fastcore.test import *
from fastlite import database
from starlette.testclient import TestClient
from fasthtml.oauth import OAuth

## PKCE and return addresses

Times are whole seconds since the epoch. A code or grant names its app by a SHA-256 digest of the app's client id:

In [ ]:
def _now(): return int(time.time())

def _digest(s): return hashlib.sha256(s.encode()).hexdigest()

An app keeps a random verifier and sends its `s256` with the sign-in. Only the app holding the verifier can then trade the code for tokens. `s256` matches the example in [RFC 7636](https://www.rfc-editor.org/rfc/rfc7636#appendix-B):

In [ ]:
def s256(verifier):
    "PKCE's S256 challenge for `verifier`"
    return base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b'=').decode()

In [ ]:
test_eq(s256('dBjftJeZ4CVP-mB92K27uhbUJU1p1r_wW1gFWFOEjXk'), 'E9Melhoa2OwvFrEMTJguCHaoeK1t8URWbuGJSstw-cM')

An app registers the addresses the person may be sent back to. Each must use https, or plain http on a loopback address. A native app such as Claude Code waits on a loopback port that it picks each time. `redirect_ok` checks an address against an app's registered addresses. A loopback address matches on any port:

In [ ]:
def loopback(uri):
    "Whether `uri` is plain http on this computer"
    s = urlsplit(uri)
    return s.scheme == 'http' and s.hostname in ('localhost', '127.0.0.1', '::1')

def redirect_ok(uris, uri):
    "Whether `uri` is one of the registered return addresses `uris`, with any port on a loopback address"
    if uri in uris: return True
    s = urlsplit(uri)
    return loopback(uri) and any(loopback(r) and (urlsplit(r).hostname, urlsplit(r).path) == (s.hostname, s.path) for r in uris)

In [ ]:
native = ['http://localhost:33418/callback']
assert redirect_ok(native, 'http://localhost:51234/callback')
assert not redirect_ok(native, 'http://localhost:51234/other')
assert not redirect_ok(native, 'https://localhost:51234/callback')

## The Allow page

The Allow page asks the signed-in person whether to let an app act as them. `AllowPage` is its default content: the app's name, where the person goes back to, and a form with Allow and Deny buttons. To use its own layout and wording, such as a list of what a connected app can do, an app passes its own `allow_page` to `OAuthServer`:

In [ ]:
def AllowPage(req, app_name, where, form):
    "A plain page asking the signed-in person to allow `app_name`"
    return Titled(f'Connect {app_name}?', P(f'{app_name} wants to act as you on this site.'),
        P(f'Only allow this if you just asked {app_name} to connect. Either way, you go back to {where}.'), form)

In [ ]:
AllowPage(None, 'Claude', 'claude.ai', Form(Button('Allow'), Button('Deny')))

(<title>Connect Claude?</title>,
 <main class="container"><h1>Connect Claude?</h1><p>Claude wants to act as you on this site.</p><p>Only allow this if you just asked Claude to connect. Either way, you go back to claude.ai.</p><form enctype="multipart/form-data"><button>Allow</button><button>Deny</button></form></main>)

## The server

`OAuthServer` adds these routes to `app`:

- `/.well-known/oauth-protected-resource`, also with `resource` appended, and `/.well-known/oauth-authorization-server` describe the server to apps.
- `/oauth/register` registers an app.
- `/oauth/authorize` shows the Allow page and takes its form.
- `/oauth/token` gives out tokens.
- `/oauth/revoke` takes a token back, when there is a `store`.

`current_user` receives the session and tells the server who is signed in. `skip` lists the server's paths and `resource`. Apps call these paths without a session. Add them to the `skip` list of the app's sign-in Beforeware, such as `OAuth`'s.

In [ ]:
class OAuthServer:
    "An OAuth 2.1 server that lets apps such as Claude act as a signed-in user at `resource`"
    def __init__(self,
        app, # The `FastHTML` app to add the server's routes to
        current_user, # `current_user(session)` returns the signed-in user's id, or `None`
        store=None, # Where grants live, such as a `FastliteStore`; adds refresh tokens, single-use codes and disconnecting apps
        resource='/mcp', # The path that access tokens work for
        login_path='/login', # Where a signed-out visitor goes; the way back is in `session['next']`
        allow_page=AllowPage, # `allow_page(req, app_name, where, form)` returns the Allow page's content
        token_user=None, # `token_user(uid, issued_at)` returns who a token acts as, or `None` to refuse it
        base_url=None, # The site's public origin, such as `https://example.com`; worked out from each request if `None`
        code_ttl=60, # Seconds a one-time code lasts
        access_ttl=None, # Seconds an access token lasts: an hour with a `store`, otherwise 30 days
        refresh_ttl=30*24*3600, # Seconds a refresh token lasts, when there is a `store`
    ):
        store_attr()
        if access_ttl is None: self.access_ttl = 3600 if store else 30*24*3600
        self.signer = URLSafeTimedSerializer(app.secret_key)
        self.grant_types = ['authorization_code', 'refresh_token'] if store else ['authorization_code']
        meta = '/.well-known/oauth-protected-resource'
        self.skip = [meta, meta+resource, '/.well-known/oauth-authorization-server', '/oauth/register', '/oauth/authorize', '/oauth/token', resource]
        app.get(meta, name='oauth_resource')(self.resource_metadata)
        app.get(meta+resource, name='oauth_resource_path')(self.resource_metadata)
        app.get('/.well-known/oauth-authorization-server', name='oauth_server')(self.server_metadata)
        app.post('/oauth/register', name='oauth_register')(self.register)
        app.get('/oauth/authorize', name='oauth_authorize')(self.authorize)
        app.post('/oauth/authorize', name='oauth_allow')(self.allow)
        app.post('/oauth/token', name='oauth_token')(self.token)
        if store:
            self.skip.append('/oauth/revoke')
            app.post('/oauth/revoke', name='oauth_revoke')(self.revoke)

`_sign` signs data with the app's secret key, with a different salt for each kind of value. A code can't pass as an access token, nor a client id as a code. `_unsign` checks the signature and the age. It returns `(None, None)` for a forged or expired value:

In [ ]:
@patch
def _sign(self:OAuthServer, kind, **d):
    "`d` signed with the app's secret key, usable only as `kind`"
    return self.signer.dumps(d, salt=f'oauth-{kind}')

@patch
def _unsign(self:OAuthServer, kind, s, max_age=None):
    "The data signed into `s` as `kind` and when it was signed, or `(None, None)` when forged or older than `max_age` seconds"
    try: d, ts = self.signer.loads(s, salt=f'oauth-{kind}', max_age=max_age, return_timestamp=True)
    except BadData: return None, None
    return d, int(ts.timestamp())

`url` gives the public address of a path. It uses `base_url` when set. Otherwise `redir_url` works it out from the request. A protected route answers a request without a valid token with `unauthorized`. Its `WWW-Authenticate` header points to the resource metadata, where an app learns how to sign in:

In [ ]:
@patch
def url(self:OAuthServer, req, path=''):
    "The public address of `path` on this site"
    return self.base_url.rstrip('/')+path if self.base_url else redir_url(req, path)

@patch
def unauthorized(self:OAuthServer, req):
    "A 401 that points an app to the resource metadata"
    meta = f'Bearer resource_metadata="{self.url(req, "/.well-known/oauth-protected-resource")}"'
    if 'authorization' in req.headers: meta += ', error="invalid_token"'
    return Response('unauthorized', status_code=401, headers={'WWW-Authenticate': meta})

The resource metadata (RFC 9728) names this site as the server that signs people in for `resource`. The server metadata (RFC 8414) lists the server's endpoints and grant types. It names the refresh token grant and the revocation endpoint only when there is a `store`:

In [ ]:
@patch
def resource_metadata(self:OAuthServer, req):
    "RFC 9728: `resource` is protected, and this site signs people in for it"
    return dict(resource=self.url(req, self.resource), authorization_servers=[self.url(req)], bearer_methods_supported=['header'])

@patch
def server_metadata(self:OAuthServer, req):
    "RFC 8414: where an app registers, sends people to allow it, and gets tokens"
    b = self.url(req)
    d = dict(issuer=b, authorization_endpoint=f'{b}/oauth/authorize', token_endpoint=f'{b}/oauth/token', registration_endpoint=f'{b}/oauth/register',
        response_types_supported=['code'], grant_types_supported=self.grant_types, code_challenge_methods_supported=['S256'], token_endpoint_auth_methods_supported=['none'])
    if self.store: d |= dict(revocation_endpoint=f'{b}/oauth/revoke', revocation_endpoint_auth_methods_supported=['none'])
    return d

`register` implements RFC 7591 registration. An app sends its name and up to 5 return addresses. The client id it gets back is the signed registration itself. Registering stores nothing. The server therefore puts no limit on registrations. Changing `secret_key` cancels every registration and token. `_err` is an OAuth error reply:

In [ ]:
def _err(code, why='', status=400): return JSONResponse(dict(error=code, error_description=why), status_code=status, headers={'Cache-Control': 'no-store'})

@patch
async def register(self:OAuthServer, req):
    "RFC 7591: register an app, replying with a client id that holds the registration"
    try: d = await req.json()
    except ValueError: return _err('invalid_client_metadata', 'Send the client metadata as JSON.')
    uris = d.get('redirect_uris') if isinstance(d, dict) else None
    if not (isinstance(uris, list) and 0 < len(uris) <= 5 and all(isinstance(u, str) and len(u) <= 300 and (u.startswith('https://') or loopback(u)) for u in uris)):
        return _err('invalid_redirect_uri', 'Send up to 5 return addresses, each using https, or http on a loopback address.')
    name = ' '.join(str(d.get('client_name') or 'An app').split())[:80]
    return JSONResponse(dict(client_id=self._sign('client', n=name, r=uris), client_id_issued_at=_now(), client_name=name, redirect_uris=uris,
        token_endpoint_auth_method='none', grant_types=self.grant_types, response_types=['code']), status_code=201)

`authorize` checks an app's sign-in link before showing anything. An unknown app or an unregistered return address gets an error page and no redirect. A redirect to that address can't be trusted. Other mistakes go back to the app as OAuth errors, through `_back`. A signed-out visitor goes to `login_path`, with the link saved in `session['next']` for the way back. Every page that `authorize` and `allow` show sends `X-Frame-Options: DENY`. Another site then can't frame the page and trick a click on Allow:

In [ ]:
def _back(uri, state, **q): return RedirectResponse(uri + ('&' if '?' in uri else '?') + urlencode(q | ({'state': state} if state else {})), status_code=303)

@patch
def _page(self:OAuthServer, *c): return HttpHeader('X-Frame-Options', 'DENY'), *c

@patch
async def authorize(self:OAuthServer, req, session, response_type:str='', client_id:str='', redirect_uri:str='', code_challenge:str='',
    code_challenge_method:str='', state:str='', resource:str=''):
    "The Allow page: check the app's link, send a signed-out visitor to sign in, and ask the person to allow the app"
    c, _ = self._unsign('client', client_id)
    if not c or not redirect_ok(c['r'], redirect_uri): return self._page(Titled('Connect an app', P('This link to connect an app is broken or out of date. Start again from the app.')))
    if response_type != 'code': return _back(redirect_uri, state, error='unsupported_response_type')
    if code_challenge_method != 'S256' or not re.fullmatch(r'[A-Za-z0-9._~-]{43,128}', code_challenge):
        return _back(redirect_uri, state, error='invalid_request', error_description='PKCE with S256 is required.')
    if resource and resource.rstrip('/') != self.url(req, self.resource):
        return _back(redirect_uri, state, error='invalid_target', error_description=f'Tokens here are only for {self.resource}.')
    if not await _arun(self.current_user(session)):
        session['next'] = f'{req.url.path}?{req.url.query}'
        return RedirectResponse(self.login_path, status_code=303)
    session['oauth_csrf'] = csrf = secrets.token_urlsafe(16)
    keep = dict(client_id=client_id, redirect_uri=redirect_uri, code_challenge=code_challenge, state=state, csrf=csrf)
    form = Form(*[Hidden(v, name=k) for k,v in keep.items()], Button('Allow'), Button('Deny', name='deny', value='1'), method='post', action='/oauth/authorize')
    where = 'an app on this computer' if loopback(redirect_uri) else urlsplit(redirect_uri).hostname
    return self._page(await _arun(self.allow_page(req, c['n'], where, form)))

`allow` takes the Allow page's form. It checks the form's nonce against the one `authorize` saved in the session. Allowing sends the app a one-time code. The code is signed and lasts `code_ttl` seconds. It holds a digest of the app's client id, the user, the return address, the PKCE challenge and a random id:

In [ ]:
@patch
async def allow(self:OAuthServer, req, session, client_id:str='', redirect_uri:str='', code_challenge:str='', state:str='', csrf:str='', deny:str=''):
    "Allow or deny the app; allowing sends it a one-time code"
    c, _ = self._unsign('client', client_id)
    uid, expected = await _arun(self.current_user(session)), session.pop('oauth_csrf', '')
    if not (c and uid and redirect_ok(c['r'], redirect_uri) and expected and secrets.compare_digest(csrf, expected)):
        return self._page(Titled('Connect an app', P('This page has expired. Start again from the app.')))
    if deny: return _back(redirect_uri, state, error='access_denied')
    code = self._sign('code', c=_digest(client_id), u=uid, r=redirect_uri, p=code_challenge, i=secrets.token_urlsafe(16))
    return _back(redirect_uri, state, code=code)

`_live` applies `token_user`. It can refuse a token, for example one issued before the user signed out of all devices. `_tokens` is the token reply from RFC 6749:

In [ ]:
@patch
async def _live(self:OAuthServer, uid, issued_at):
    "Who a token for `uid` issued at `issued_at` acts as, or `None` when `token_user` refuses it"
    return await _arun(self.token_user(uid, issued_at)) if self.token_user else uid

@patch
def _tokens(self:OAuthServer, access, refresh=None):
    "RFC 6749's token reply"
    d = dict(access_token=access, token_type='Bearer', expires_in=self.access_ttl) | (dict(refresh_token=refresh) if refresh else {})
    return JSONResponse(d, headers={'Cache-Control': 'no-store'})

`_redeem` trades a one-time code and its PKCE verifier for tokens. Without a store it returns a signed access token for the user. It can't tell whether the code was used before. With a store, the code's random id becomes the id of a new grant. A second redemption finds that id taken. The server then refuses it and revokes the grant, as RFC 6749 asks. A new grant also revokes the user's older grants to the same app:

In [ ]:
@patch
async def _redeem(self:OAuthServer, client_id, code, verifier, redirect_uri):
    "Trade a one-time code and its PKCE verifier for tokens"
    c, _ = self._unsign('client', client_id)
    k, _ = self._unsign('code', code, self.code_ttl)
    if not (c and k) or k['c'] != _digest(client_id) or k['r'] != redirect_uri: return _err('invalid_grant', 'The code is wrong or out of date.')
    if not secrets.compare_digest(s256(verifier), k['p']): return _err('invalid_grant', 'The PKCE verifier does not match.')
    if not self.store: return self._tokens(self._sign('access', u=k['u']))
    if not await _arun(self.store.add_grant(dict(id=k['i'], client=k['c'], name=c['n'], uid=k['u'], n=0, created_at=_now(), revoked=False))):
        await _arun(self.store.revoke(k['i']))
        return _err('invalid_grant', 'The code was already used, so its tokens no longer work.')
    for old in await _arun(self.store.grants(k['u'])):
        if old.client == k['c'] and old.id != k['i']: await _arun(self.store.revoke(old.id))
    return self._tokens(self._sign('access', g=k['i'], u=k['u']), self._sign('refresh', g=k['i'], n=0))

With a store, `_refresh` trades a refresh token for a new pair. Each refresh token carries its grant's counter. `advance` moves the counter on only from that value. A replayed refresh token fails that check and revokes its grant. `token` sends each grant type to its handler:

In [ ]:
@patch
async def _refresh(self:OAuthServer, client_id, refresh_token):
    "Trade a refresh token for a new pair; replaying a used one revokes its grant"
    r, issued = self._unsign('refresh', refresh_token, self.refresh_ttl)
    g = await _arun(self.store.grant(r['g'])) if r else None
    if not g or g.revoked or g.client != _digest(client_id): return _err('invalid_grant', 'The refresh token is unknown, revoked or out of date.')
    if not await _arun(self.store.advance(g.id, r['n'])):
        await _arun(self.store.revoke(g.id))
        return _err('invalid_grant', 'This refresh token was already used, so the app has been disconnected.')
    if not await self._live(g.uid, issued):
        await _arun(self.store.revoke(g.id))
        return _err('invalid_grant', 'This sign-in no longer works. Connect the app again.')
    return self._tokens(self._sign('access', g=g.id, u=g.uid), self._sign('refresh', g=g.id, n=r['n']+1))

@patch
async def token(self:OAuthServer, req, grant_type:str='', client_id:str='', code:str='', code_verifier:str='', redirect_uri:str='', refresh_token:str='', resource:str=''):
    "Trade a one-time code and its PKCE verifier, or with a store a refresh token, for new tokens"
    if resource and resource.rstrip('/') != self.url(req, self.resource): return _err('invalid_target', f'Tokens here are only for {self.resource}.')
    if grant_type == 'authorization_code': return await self._redeem(client_id, code, code_verifier, redirect_uri)
    if grant_type == 'refresh_token' and self.store: return await self._refresh(client_id, refresh_token)
    return _err('unsupported_grant_type')

With a store, `revoke` implements RFC 7009. An app gives up a token through it. Giving up a token revokes its grant. Revoking an unknown token isn't an error:

In [ ]:
@patch
async def revoke(self:OAuthServer, token:str=''):
    "RFC 7009: an app gives up a token, which revokes its grant; an unknown token is no error"
    d = self._unsign('refresh', token)[0] or self._unsign('access', token)[0]
    if d: await _arun(self.store.revoke(d['g']))
    return Response(status_code=200)

A protected route calls `user` to find who a request's access token acts as. With a store, `user` also checks that the token's grant hasn't been revoked. `apps` lists a user's live grants. `disconnect` revokes one of them, if it belongs to that user. Without a store, both raise `NotImplementedError`:

In [ ]:
@patch
async def user(self:OAuthServer, req):
    "Who the request's access token acts as, or `None`"
    hdr = req.headers.get('authorization', '')
    a, issued = self._unsign('access', hdr[7:], self.access_ttl) if hdr.startswith('Bearer ') else (None, None)
    if not a: return None
    if self.store and not ((g := await _arun(self.store.grant(a['g']))) and not g.revoked): return None
    return await self._live(a['u'], issued)

@patch
async def apps(self:OAuthServer, uid):
    "The grants `uid` has made, newest first, each with the app's `name` and the `id` that `disconnect` takes"
    if not self.store: raise NotImplementedError('Listing connected apps needs a store')
    return await _arun(self.store.grants(uid))

@patch
async def disconnect(self:OAuthServer, uid, grant_id):
    "Revoke grant `grant_id` if `uid` made it, ending its app's tokens"
    if not self.store: raise NotImplementedError('Disconnecting apps needs a store')
    if (g := await _arun(self.store.grant(grant_id))) and g.uid == uid: await _arun(self.store.revoke(grant_id))

## Grants

A store holds grants. A grant records that a user connected an app: the app's name and client id digest, the user, a refresh counter, and whether it's revoked. A store needs five methods:

- `add_grant(grant)` adds a grant, returning `False` if its id exists already.
- `grant(grant_id)` returns a grant, or `None`.
- `advance(grant_id, n)` moves a live grant's counter from `n` to `n+1`, returning `False` if it isn't at `n`.
- `revoke(grant_id)` revokes a grant.
- `grants(uid)` lists a user's live grants, newest first.

`add_grant` and `advance` must each be one atomic step. Two requests then can't both succeed, even from different processes. A store's methods can be sync or async. The server calls each one through `_arun`.

`FastliteStore` keeps grants in a [fastlite](https://fastlite.answer.ai) database that the app creates. FastHTML doesn't depend on fastlite. Install it to use `FastliteStore`, as for `fast_app`'s `db_file`:

In [ ]:
class OAuthGrant: id:str; client:str; name:str; uid:str; n:int; created_at:int; revoked:bool

class FastliteStore:
    "Grants in fastlite database `db`, one row for each app a user has connected"
    def __init__(self, db):
        self.db = db
        self.grants_t = db.create(OAuthGrant, name='oauth_grant', pk='id', if_not_exists=True)
    def add_grant(self, g):
        sql = 'INSERT INTO oauth_grant (id, client, name, uid, n, created_at, revoked) VALUES (?, ?, ?, ?, ?, ?, ?) ON CONFLICT(id) DO NOTHING RETURNING id'
        return bool(self.db.q(sql, [g['id'], g['client'], g['name'], g['uid'], g['n'], g['created_at'], g['revoked']]))
    def grant(self, grant_id): return self.grants_t.get(grant_id, default=None)
    def advance(self, grant_id, n): return bool(self.db.q('UPDATE oauth_grant SET n=n+1 WHERE id=? AND n=? AND NOT revoked RETURNING id', [grant_id, n]))
    def revoke(self, grant_id): self.db.execute('UPDATE oauth_grant SET revoked=1 WHERE id=?', [grant_id])
    def grants(self, uid): return self.grants_t('uid=? AND NOT revoked', [uid], order_by='created_at DESC')

A grant id can be added once. Its counter moves on once from each value:

In [ ]:
store = FastliteStore(database(':memory:'))
g = dict(id='g1', client='c', name='Claude', uid='alice', n=0, created_at=_now(), revoked=False)
assert store.add_grant(g) and not store.add_grant(g)
assert store.advance('g1', 0) and not store.advance('g1', 0)

## Connecting an app

`mk_app` makes an app whose `/login` signs the visitor in as `alice` and returns to the page saved in `session['next']`. Its route at `/mcp` stands in for an MCP server. It answers with the user that a token acts as. Without a valid token it returns `unauthorized`. This first app has no store:

In [ ]:
def mk_app(store=None):
    "An `OAuthServer` app with a sign-in at `/login` and a protected route at `/mcp`, and a client for it"
    app = FastHTML(secret_key='soopersecret')
    srv = OAuthServer(app, current_user=lambda session: session.get('auth'), store=store)
    @app.get('/login')
    def login(session):
        session['auth'] = 'alice'
        return RedirectResponse(session.pop('next', '/'), status_code=303)
    @app.post('/mcp')
    async def mcp(req):
        if not (uid := await srv.user(req)): return srv.unauthorized(req)
        return dict(user=uid)
    return srv, TestClient(app)

srv, cli = mk_app()

Without a token, `/mcp` answers 401 and names the resource metadata:

In [ ]:
r = cli.post('/mcp')
test_eq(r.status_code, 401)
r.headers['www-authenticate']

'Bearer resource_metadata="https://testserver/.well-known/oauth-protected-resource"'

The resource metadata names the server that signs people in. Without a store, the server metadata offers only the code grant:

In [ ]:
meta = cli.get('/.well-known/oauth-protected-resource/mcp').json()
test_eq(cli.get('/.well-known/oauth-authorization-server').json()['grant_types_supported'], ['authorization_code'])
meta

{'resource': 'https://testserver/mcp',
 'authorization_servers': ['https://testserver'],
 'bearer_methods_supported': ['header']}

Claude registers with the address claude.ai sends people back to. Its client id holds the registration:

In [ ]:
cb = 'https://claude.ai/api/mcp/auth_callback'
r = cli.post('/oauth/register', json=dict(client_name='Claude', redirect_uris=[cb]))
test_eq(r.status_code, 201)
claude = r.json()
claude

{'client_id': 'eyJuIjoiQ2xhdWRlIiwiciI6WyJodHRwczovL2NsYXVkZS5haS9hcGkvbWNwL2F1dGhfY2FsbGJhY2siXX0.av0eEw.c3rIhBYXHkR0F47E2ZMH4wvv8ec',
 'client_id_issued_at': 1794973203,
 'client_name': 'Claude',
 'redirect_uris': ['https://claude.ai/api/mcp/auth_callback'],
 'token_endpoint_auth_method': 'none',
 'grant_types': ['authorization_code'],
 'response_types': ['code']}

An address that is neither https nor loopback is refused:

In [ ]:
test_eq(cli.post('/oauth/register', json=dict(redirect_uris=['http://example.com/cb'])).json()['error'], 'invalid_redirect_uri')

Claude's sign-in link carries its client id, its return address, a PKCE challenge, its `state` and the resource it wants tokens for:

In [ ]:
verifier = secrets.token_urlsafe(48)
def sign_in_link(claude):
    "Claude's link to the Allow page"
    q = dict(response_type='code', client_id=claude['client_id'], redirect_uri=cb, code_challenge=s256(verifier), code_challenge_method='S256', state='xyz', resource=meta['resource'])
    return f'/oauth/authorize?{urlencode(q)}'
auth_url = sign_in_link(claude)

A signed-out visitor is sent to sign in, then back to the same link:

In [ ]:
test_eq(cli.get(auth_url, follow_redirects=False).headers['location'], '/login')
test_eq(cli.get('/login', follow_redirects=False).headers['location'], auth_url)

Signed in, the person sees the Allow page. No other site may show it in a frame:

In [ ]:
r = cli.get(auth_url)
test_eq(r.headers['x-frame-options'], 'DENY')
assert 'Connect Claude?' in r.text and 'claude.ai' in r.text

`press_allow` posts the Allow page's form. It returns the query that Claude gets back:

In [ ]:
def press_allow(cli, url):
    "Press Allow on the Allow page at `url`, and return the query the app gets back"
    fields = {i['name']: i['value'] for i in BeautifulSoup(cli.get(url).text, 'html.parser').select('form input')}
    r = cli.post('/oauth/authorize', data=fields, follow_redirects=False)
    return dict(parse_qsl(urlsplit(r.headers['location']).query))

back = press_allow(cli, auth_url)
test_eq(back['state'], 'xyz')
back.keys()

dict_keys(['code', 'state'])

A link naming an unknown app gets the error page, with no redirect. A sign-in without S256 goes back to the app with an OAuth error:

In [ ]:
r = cli.get(auth_url.replace(claude['client_id'], 'nope'), follow_redirects=False)
assert r.status_code == 200 and 'broken or out of date' in r.text
r = cli.get(auth_url.replace('code_challenge_method=S256', 'code_challenge_method=plain'), follow_redirects=False)
assert r.headers['location'].startswith(f'{cb}?error=invalid_request')

Claude trades the code and its verifier for an access token. Without a store there is no refresh token:

In [ ]:
tok_req = dict(grant_type='authorization_code', client_id=claude['client_id'], redirect_uri=cb, code_verifier=verifier)
toks = cli.post('/oauth/token', data=tok_req | dict(code=back['code'])).json()
toks.keys()

dict_keys(['access_token', 'token_type', 'expires_in'])

A code redeemed with the wrong verifier gives no tokens:

In [ ]:
code = press_allow(cli, auth_url)['code']
test_eq(cli.post('/oauth/token', data=tok_req | dict(code=code, code_verifier='x'*43)).json()['error_description'], 'The PKCE verifier does not match.')

With the access token, `/mcp` runs as `alice`:

In [ ]:
def call_mcp(cli, tok): return cli.post('/mcp', headers={'Authorization': f'Bearer {tok}'})
test_eq(call_mcp(cli, toks['access_token']).json(), {'user': 'alice'})

Without a store, `apps` raises `NotImplementedError`. `token_user` is the only way to end tokens early, for example ones issued before the user last signed out of all devices:

In [ ]:
with expect_fail(NotImplementedError): await srv.apps('alice')
signed_out = _now()+1
srv.token_user = lambda uid, issued_at: uid if issued_at >= signed_out else None
test_eq(call_mcp(cli, toks['access_token']).status_code, 401)
srv.token_user = None

## Connecting with a store

This app keeps grants in a `FastliteStore`. `connect` signs `alice` in, presses Allow for Claude and redeems the code. It returns the token reply and the request that redeemed the code:

In [ ]:
srv, cli = mk_app(FastliteStore(database(':memory:')))
claude = cli.post('/oauth/register', json=dict(client_name='Claude', redirect_uris=[cb])).json()
cli.get('/login')

def connect():
    "Connect Claude as `alice`, returning the token reply and the code's token request"
    req = dict(grant_type='authorization_code', client_id=claude['client_id'], redirect_uri=cb, code_verifier=verifier, code=press_allow(cli, sign_in_link(claude))['code'])
    return cli.post('/oauth/token', data=req).json(), req

Claude now gets a refresh token too. The code still works only once. A second redemption revokes its grant. The first access token then stops working:

In [ ]:
toks, req = connect()
assert 'refresh_token' in toks
test_eq(cli.post('/oauth/token', data=req).json()['error'], 'invalid_grant')
test_eq(call_mcp(cli, toks['access_token']).status_code, 401)

A refresh token gives one new pair. Using it again revokes the grant:

In [ ]:
toks, _ = connect()
ref = dict(grant_type='refresh_token', client_id=claude['client_id'], refresh_token=toks['refresh_token'])
new = cli.post('/oauth/token', data=ref).json()
test_eq(call_mcp(cli, new['access_token']).status_code, 200)
test_eq(cli.post('/oauth/token', data=ref).json()['error'], 'invalid_grant')
test_eq(call_mcp(cli, new['access_token']).status_code, 401)

`apps` lists Claude's live grant. Connecting again replaces it:

In [ ]:
first_toks, _ = connect()
toks, _ = connect()
grants = await srv.apps('alice')
test_eq([g.name for g in grants], ['Claude'])
test_eq(call_mcp(cli, first_toks['access_token']).status_code, 401)

`disconnect` revokes the grant. Claude's access token stops working at once:

In [ ]:
test_eq(call_mcp(cli, toks['access_token']).status_code, 200)
await srv.disconnect('alice', grants[0].id)
test_eq(await srv.apps('alice'), [])
test_eq(call_mcp(cli, toks['access_token']).status_code, 401)

## Signing in with `OAuth`

When FastHTML's `OAuth` handles sign-in, pass the server's paths in its `skip`. After sign-in, `OAuth` returns to the Allow page saved in `session['next']`. Here `StubClient` stands in for a provider such as Google. It signs everyone in as `bob`:

In [ ]:
class StubClient:
    "A provider that signs everyone in as `bob`"
    id_key = 'sub'
    async def retr_info_async(self, code, redirect_uri): return dict(sub='bob')

class Auth(OAuth):
    def get_auth(self, info, ident, session, state): return RedirectResponse('/', status_code=303)

app2 = FastHTML(secret_key='soopersecret')
srv2 = OAuthServer(app2, current_user=lambda session: session.get('auth'))
oauth = Auth(app2, StubClient(), skip=[*srv2.skip, '/redirect', '/error', '/login'])
cli2 = TestClient(app2)

A signed-out visitor goes to `/login`. After the provider sends them back to `/redirect`, `OAuth` returns them to the Allow page:

In [ ]:
link = sign_in_link(cli2.post('/oauth/register', json=dict(client_name='Claude', redirect_uris=[cb])).json())
test_eq(cli2.get(link, follow_redirects=False).headers['location'], '/login')
test_eq(cli2.get('/redirect?code=abc', follow_redirects=False).headers['location'], link)
assert 'Connect Claude?' in cli2.get(link).text

# Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()